# 02 - Exploratory Factor Analysis
This notebook calculates the overall historical return, risk and other characteristics of DM factor-indices.

## 1. Load and check processed data

In [ ]:
# Import libraries

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Display settings
pd.set_option("display.max_columns", 10)
pd.set_option("display.float_format", lambda value: f"{value:.4f}")

In [ ]:
# Define the path to the data directory

current_directory = Path.cwd()

if current_directory.name == "notebooks":
    repository_root = current_directory.parent
else:
    repository_root = current_directory

processed_data_directory = repository_root / "data" / "processed"

print("Repository root:", repository_root)
print("Processed data directory:", processed_data_directory)

In [ ]:
# Set data directory path

monthly_levels_path = (processed_data_directory / "monthly_index_levels.csv")

monthly_returns_full_path = (processed_data_directory / "monthly_returns_full.csv")

monthly_returns_common_path = (processed_data_directory / "monthly_returns_common.csv")

In [ ]:
# Check if the files exist

print("Monthly levels file exists:", monthly_levels_path.exists())
print("Monthly returns full file exists:", monthly_returns_full_path.exists())
print("Monthly returns common file exists:", monthly_returns_common_path.exists())

In [ ]:
# Load the processed data

monthly_returns = pd.read_csv(monthly_returns_common_path, index_col="date", parse_dates=True)


In [ ]:
# Check shape

print("Shape:", monthly_returns.shape)
print("Columns:", monthly_returns.columns.tolist())
print("Index type", type(monthly_returns.index))

In [ ]:
# Check Timeframe 

print("First Observation:", monthly_returns.index.min())
print("Last Observations:", monthly_returns.index.max())
print("Number of months:", len(monthly_returns))
print("Number of series:", monthly_returns.shape[1])

In [ ]:
# Data validation

print("Missing values:", monthly_returns.isna().sum().sum())
print("Duplicate dates:", monthly_returns.index.duplicated().sum())
print("Returns <= -100%", (monthly_returns <= -1).sum().sum())
print("Non-numeric columns", monthly_returns.select_dtypes(exclude="number").columns.tolist())

In [ ]:
# More tests

assert monthly_returns.index.is_monotonic_increasing
assert not monthly_returns.index.duplicated().any()
assert not monthly_returns.isna().any().any()
assert (monthly_returns > -1).all().all()

## 2. Static analysis

In [ ]:
# Show structure

monthly_returns.info()

In [ ]:
# Overall range of returns

return_ranges = pd.DataFrame({"minimum": monthly_returns.min(), 
                               "maximum": monthly_returns.max(),
                               "mean": monthly_returns.mean()})

return_ranges

In [ ]:
# Monthly statistics

monthly_statistics = pd.DataFrame({"mean_monthly_return": monthly_returns.mean(),
                                   "monthly_volatility": monthly_returns.std(),
                                   "minimum_monthly_return": monthly_returns.min(),
                                   "maximum_monthly_return": monthly_returns.max(),
                                   "positive_month_share": (monthly_returns > 0).mean(),
                                   "skewness": monthly_returns.skew(),
                                   "excess_kurtosis": monthly_returns.kurt()
})

monthly_statistics

### 2.1 Annualized geometric return

$$ r_{\mathrm{annualized}} = \bigg( \prod_{t=1}^{T} (1 + r_t) \bigg)^{12/T} - 1 $$

where $ r_t $ are the simple periodic returns

In [ ]:
# Define annualized geometric returns function

def annualized_geometric_returns(
        returns,
        periods_per_year = 12
): 
    returns = returns.dropna()
    number_of_periods = len(returns)
    total_growth = (1 + returns).prod()

    annualized_return = (total_growth**(periods_per_year / number_of_periods) - 1)

    return annualized_return

In [ ]:
# Test the function

market_annualized_return = (annualized_geometric_returns(monthly_returns["market"]))

market_annualized_return

### 2.2 Annualized volatility

$$\sigma_{\mathrm{annualized}} = \sigma_{\mathrm{monthly}}\sqrt{12}$$

In [ ]:
# Annualized volatility

annualized_volatility = (monthly_returns.std()*np.sqrt(12))

annualized_volatility

In [ ]:
# Summary

summary_statistics = pd.DataFrame({
    "annualized_return": monthly_returns.apply(annualized_geometric_returns),
    "annualized_volatility": annualized_volatility,
    "mean_monthly_returns": monthly_returns.mean(),
    "minimum_monthly_return": monthly_returns.min(),
    "maximum_monthly_returns": monthly_returns.max(),
    "positive_month_share": (monthly_returns > 0).mean(),
    "skewness": monthly_returns.skew(),
    "excess_kurtosis": monthly_returns.kurt()
})

summary_statistics

### 2.3 Wealth index

$$W_t = \prod_{i=1}^{t} (1 + r_i)$$

where $r_i \hat{=}$ monthly returns

In [ ]:
# Wealth index

wealth_index = (1 + monthly_returns).cumprod()

wealth_index

In [ ]:
# Plot wealth index

fig, ax = plt.subplots(figsize=(12,6))

wealth_index.plot(ax=ax)
ax.set_title("Growth of one unit each")
ax.set_xlabel("Date")
ax.set_ylabel("Wealth Index")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Logartihmic plot

fig, ax = plt.subplots(figsize=(12,6))

wealth_index.plot(ax=ax, logy=True)
ax.set_title("Logarithmic growth of one unit each")
ax.set_xlabel("Date")
ax.set_ylabel("Wealth Index (Logarithmic)")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.plot()

### 2.4 Relative performance of factor indices to the broad market 

$$RW_{t}^{i} = \frac{W_{t}^{i}}{W_{t}^{\mathrm{Market}}}$$

$RW \hat{=}$ Relative Weatlth
for $i \in$ {Value, Momentum, Quality, Min_Volatility, Size_Proxy}

In [ ]:
# Calculate relative wealth

relative_wealth = wealth_index.div(wealth_index["market"], axis="index")

relative_wealth = relative_wealth.drop(columns="market")

relative_wealth

In [ ]:
# Plot relative wealth

fig, ax = plt.subplots(figsize=(12,6))

relative_wealth.plot(ax=ax)
ax.axhline(y=1, linestyle="--")
ax.set_title("Cumulative Performance of each factor relative to the market")
ax.set_xlabel("Date")
ax.set_ylabel("Relative Wealth")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.plot()

### 2.5 Correlation matrix

$$\rho_{i,j} = \frac{\mathrm{cov}(r_i , r_j)}{\sigma_i \sigma_j}$$

where $\sigma_{i,j}$ are the index's volatilities and $r_{i,j}$ the index's returns

In [ ]:
# Calculate correlation matrix

correlation_matrix = monthly_returns.corr()

correlation_matrix

In [ ]:
# Heatmap

fig, ax = plt.subplots(figsize=(8,6))

image = ax.imshow(correlation_matrix, vmin=-1, vmax=1)

ax.set_xticks(range(len(correlation_matrix.columns)))

ax.set_xticklabels(correlation_matrix.columns, rotation=45, ha="right")

ax.set_yticks(range(len(correlation_matrix.index)))

ax.set_yticklabels(correlation_matrix.index)

for row in range(len(correlation_matrix.index)):
    for column in range(len(correlation_matrix.columns)):
        value = correlation_matrix.iloc[row, column]

        ax.text(column, row, f"{value:.2f}", ha="center", va="center")

fig.colorbar(image, ax=ax, label="Correlation")
ax.set_title("Correlation of monthly factor index returns")
plt.tight_layout()
plt.plot()

In [ ]:
# Avarage correlation of each factor

avarage_correlations = (correlation_matrix.apply(lambda column: column.drop(labels = column.name).mean()).sort_values)

avarage_correlations

### 2.6 Drawdown analysis

$$D_t = \frac{W_t}{\mathrm{max}_{\tau \leq t}(W_\tau)} - 1$$

we calculate the drawdowns $D_t$ realtive to the historic maximum before $W_\tau$ using the wealth index

In [ ]:
# Calculate maximum drawdown

running_maximum = wealth_index.cummax()

drawdowns = (wealth_index / running_maximum -1)

maximum_drawdowns = drawdowns.min()

# Add them to the summary of statistics

summary_statistics["maximum_drawdown"] = maximum_drawdowns

summary_statistics["maximum_drawdown"]

### 2.7 Calmar Ratio

$$\mathrm{Calmar} = \frac{r_{\mathrm{annualized}}}{|D_{\mathrm{max}}|}$$

where $D_{\mathrm{max}}$ is the index's maximum drawdown and $r_{\mathrm{annualized}}$ is the calculated annualized return in 2.1

In [ ]:
# Calmar Ratio

calmar_ratio = (summary_statistics["annualized_return"] / summary_statistics["maximum_drawdown"].abs())

# Add them to the summary of statistics

summary_statistics["calmar_ratio"] = calmar_ratio

summary_statistics["calmar_ratio"]

### 2.8 Sharpe Ratio

$$\mathrm{Sharpe}=\frac{R_\rho - R_f}{\sigma_p}$$

where 
$R_\rho \hat{=}$ return of index
$R_f \hat{=}$ risk free rate
$\sigma_p \hat{=}$ standard deviation of the index's excess return

we assume $r_f = 0$ the annualized Sharpe Ratio becomes

$$\mathrm{Sharpe}_{\mathrm{annualized}}=\frac{\overline{r_{\mathrm{month}}}}{\sigma_{\mathrm{month}}}\sqrt{12}$$

where we use the avarage monthly return $\overline{r_{\mathrm{month}}}$

In [ ]:
# Sharpe Ratio

sharpe_ratio_zero_rf = (monthly_returns.mean() / monthly_returns.std())*np.sqrt(12)

summary_statistics["sharpe_ratio_zero_rf"] = sharpe_ratio_zero_rf

summary_statistics["sharpe_ratio_zero_rf"]

### 2.9 Sortino Ratio

$$\mathrm{Sortino} = \frac{R_\rho - R_f}{\sigma_d}$$

where $\sigma_d$ is the index's downside deviation

In [ ]:
# Sortino Ratio

def annualized_downside_deviation(returns, target_return=0) : 
    downside_returns = np.minimum(returns - target_return, 0)

    mean_squared_downside = (downside_returns.pow(2).mean())
    return ( np.sqrt(mean_squared_downside)*np.sqrt(12))

annualized_downside_risk = (monthly_returns.apply(annualized_downside_deviation))

annualized_mean_return = (monthly_returns.mean()*12)

sortino_ratio_zero_target = (annualized_mean_return/annualized_downside_risk)

summary_statistics["downside_deviation"] = annualized_downside_risk
summary_statistics["sortino_ratio_zero_target"] = sortino_ratio_zero_target

summary_statistics["sortino_ratio_zero_target"]

In [ ]:
# Summary

performance_summary = summary_statistics[["annualized_return", "annualized_volatility", "maximum_drawdown", "sharpe_ratio_zero_rf", "downside_deviation", "sortino_ratio_zero_target","calmar_ratio"]].copy()

performance_summary

## 3. Rolling window analysis

Rolling annualized returns show how factor performance varies in different historical 5-year periods.

### 3.1 Rolling annualized returns

In [ ]:
# Length of the return and risk windows in months

return_window_months = 60
risk_window_months = 36

In [ ]:
# Define annualized return from window function

def annualized_return_from_window(returns, periods_per_year=12):
    number_of_periods = len(returns)
    total_growth = (1 + returns).prod()

    annualized_return = (total_growth**(periods_per_year / number_of_periods) - 1)

    return annualized_return

In [ ]:
# Calculate rolling annualized returns

rolling_annualized_returns = (monthly_returns.rolling(window=return_window_months, min_periods=return_window_months).apply(annualized_return_from_window, raw = False))

In [ ]:
# Plot rolling annualized returns

fig, ax = plt.subplots(figsize=(12, 6))

rolling_annualized_returns.plot(ax=ax)

ax.axhline(y = 0, color='black', linestyle='--', linewidth=1)

ax.set_title("Rolling Annualized Returns (60-month window)")

ax.set_xlabel("Date")
ax.set_ylabel("Annualized Return")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Share of negative 60-month rolling annualized returns

negative_rolling_returns_share = (rolling_annualized_returns.apply(lambda series: (series.dropna() < 0).mean()))
negative_rolling_returns_share


### 3.2 Rolling annualized volatility

$$ \sigma_{t} = \sigma(r_{t-T}, ...,r_t)\sqrt{12}$$

where $T$ is the rolling window (in months)


In [ ]:
# Calculate rolling annualized volatility

rolling_annualized_volatility = (monthly_returns.rolling(window=risk_window_months, min_periods=risk_window_months).std()*np.sqrt(12))

In [ ]:
# Plot rolling annualized returns

fig, ax = plt.subplots(figsize=(12, 6))

rolling_annualized_volatility.plot(ax=ax)

ax.set_title("Rolling Annualized Volatility (36-month window)")

ax.set_xlabel("Date")
ax.set_ylabel("Annualized Volatility")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Bandwidth of rolling annualized volatility

rolling_volatility_summary = pd.DataFrame({
    "minimum": rolling_annualized_volatility.min(),
    "maximum": rolling_annualized_volatility.max(),
    "median": rolling_annualized_volatility.median(),
})

rolling_volatility_summary

### 3.3 Rolling market correlation

In [ ]:
# Calculate rolling market correlations

factor_columns = [column for column in monthly_returns.columns if column != "market"]

rolling_market_correlations = pd.DataFrame(index = monthly_returns.index)

for factor in factor_columns:
    rolling_market_correlations[factor] = monthly_returns[factor].rolling(window=risk_window_months, min_periods=risk_window_months).corr(monthly_returns["market"])

In [ ]:
# Plot rolling market correlations

fig, ax = plt.subplots(figsize=(12,6))

rolling_market_correlations.plot(ax=ax)

ax.set_title("Rolling Market Correlations (36-month window)")
ax.set_xlabel("Date")
ax.set_ylabel("Correlation")
ax.set_ylim(-1, 1)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Bandwidth of rolling market correlations

rolling_market_correlation_summary = pd.DataFrame({
    "minimum": rolling_market_correlations.min(),
    "maximum": rolling_market_correlations.max(),
    "median": rolling_market_correlations.median(),
})

rolling_market_correlation_summary

### 3.4 Rolling outperformance relative to the market

For each factor we compute:

$$ R_{rel} = \frac{\prod_t(1+r_t)}{\prod_t(1 +r_{\mathrm{Market}})} -1$$

In [ ]:
# Define relative performance function 

def rolling_relative_return(factor_returns, market_returns, window):
    factor_growth = (1 + factor_returns).rolling(window).apply(np.prod, raw=True)
    market_growth = (1 + market_returns).rolling(window).apply(np.prod, raw=True)
    return (factor_growth / market_growth - 1)


In [ ]:
# Calculate relative performance of all factors relative to the market over a rolling window

rolling_window_performance = pd.DataFrame(index = monthly_returns.index)

for factor in factor_columns:
    rolling_window_performance[factor] = ( rolling_relative_return(factor_returns=monthly_returns[factor], market_returns=monthly_returns["market"], window=return_window_months))


In [ ]:
# Plot relative performance of all factors relative to the market over a rolling window

fig, ax = plt.subplots(figsize=(12,6))

rolling_window_performance.plot(ax=ax)

ax.axhline(y=0, color='black', linestyle='--', linewidth=1)

ax.set_title("Rolling Relative Performance of Factors vs Market (60-month window)")
ax.set_xlabel("Date")
ax.set_ylabel("Relative Performance")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Share of 5-year periods with outperformance relative to the market

outperformance_share = (rolling_window_performance.apply(lambda series: (series.dropna() > 0).mean()))

outperformance_share

## Conclusion

This exploratory analysis caluclates return, risk, drawdown and dependence characteristics of DM factor indices.

We work with rolling windows since full-sample performance alone may hide extended periods of factor underperfomance, changing volatility or correlations with the broad market.

The generated plots for the rolling-widnow analysis show the past 5-year (60 months) performance or 3-year risk (36 months) of each facor, therefore the graphs start 5 years after the time of the first datapoint.